In [1]:
import joblib

X_train_balanced = joblib.load('../data/X_train_balanced.pkl')
y_train_balanced = joblib.load('../data/y_train_balanced.pkl')
X_test = joblib.load('../data/X_test.pkl')
y_test = joblib.load('../data/y_test.pkl')
feature_columns = joblib.load('../data/feature_columns.pkl')

print("Training shape:", X_train_balanced.shape)
print("Testing shape:", X_test.shape)

Training shape: (8278, 19)
Testing shape: (1409, 19)


In [4]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train_balanced, y_train_balanced)
    preds = model.predict(X_test)
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1 Score': f1_score(y_test, preds)
    })

results_df = pd.DataFrame(results)
results_df

c:\Users\sIMRAN\OneDrive\Desktop\telecom_churn\myenvq\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.753016,0.525292,0.721925,0.608108
1,Random Forest,0.779276,0.584450,0.582888,0.583668
2,Decision Tree,0.716820,0.469734,0.518717,0.493011


In [5]:
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

    score = cross_val_score(model, X_train_balanced, y_train_balanced, cv=3, scoring='f1').mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("Best parameters:", study.best_params)
print("Best F1 score:", study.best_value)

[I 2026-07-23 02:00:43,301] A new study created in memory with name: no-name-a8133051-cd3c-4d0f-881a-11eef14b1063
[I 2026-07-23 02:00:44,761] Trial 0 finished with value: 0.8045640346254213 and parameters: {'n_estimators': 154, 'max_depth': 10, 'min_samples_split': 9, 'min_samples_leaf': 2}. Best is trial 0 with value: 0.8045640346254213.
[I 2026-07-23 02:00:46,333] Trial 1 finished with value: 0.7894478010872756 and parameters: {'n_estimators': 157, 'max_depth': 19, 'min_samples_split': 8, 'min_samples_leaf': 7}. Best is trial 0 with value: 0.8045640346254213.
[I 2026-07-23 02:00:48,449] Trial 2 finished with value: 0.7973472141231004 and parameters: {'n_estimators': 237, 'max_depth': 10, 'min_samples_split': 3, 'min_samples_leaf': 6}. Best is trial 0 with value: 0.8045640346254213.
[I 2026-07-23 02:00:49,990] Trial 3 finished with value: 0.7922598085787134 and parameters: {'n_estimators': 176, 'max_depth': 10, 'min_samples_split': 8, 'min_samples_leaf': 9}. Best is trial 0 with value

Best parameters: {'n_estimators': 264, 'max_depth': 9, 'min_samples_split': 2, 'min_samples_leaf': 1}
Best F1 score: 0.8054491755193092


In [6]:
import joblib

# Train final model with the best parameters found
best_model = RandomForestClassifier(**study.best_params, random_state=42)
best_model.fit(X_train_balanced, y_train_balanced)

# Evaluate on test set
final_preds = best_model.predict(X_test)
print("Final Accuracy:", accuracy_score(y_test, final_preds))
print("Final F1 Score:", f1_score(y_test, final_preds))

# Save the model for later use in the Streamlit app
joblib.dump(best_model, '../models/churn_model.pkl')
joblib.dump(feature_columns, '../models/feature_columns.pkl')

print("Model saved successfully!")

Final Accuracy: 0.7700496806245565
Final F1 Score: 0.6170212765957447
Model saved successfully!
